# Notebook 6 — Experiments
**MSML 604 — Sparse Factor Models for Equity Return Prediction**

---

## Overview
This notebook runs all experiments and documents every finding.
Each experiment answers a specific research question.

| Experiment | Research Question |
|---|---|
| 6.1 Cross-validation | What is the optimal lambda for each method? |
| 6.2 Factor dropout table | Which factors are consistently selected? |
| 6.3 Regime analysis | Does adaptive lambda help during crises? |
| 6.4 Online vs Batch | Is online learning competitive with batch? |
| 6.5 Factor interactions | Do interaction terms improve prediction? |
| 6.6 DRO analysis | Does robust optimization help? |
| 6.7 Signal discovery | Can raw price signals match FF factors? |

In [2]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from scipy.stats import spearmanr
sys.path.insert(0, '..')
from src.data_loader import load_all_data
from src.solvers import RidgeScratch, LassoProximal, ElasticNetScratch
from src.backtest import walk_forward_backtest, compute_metrics
from src.online_solver import walk_forward_online
from src.cross_validation import find_best_alphas, time_series_cv
from src.regime import identify_regimes, compute_adaptive_lambda
from src.backtest import walk_forward_backtest_adaptive
from src.interactions import (build_interaction_features,
                               analyze_interaction_selection)
from src.dro_solver import DROLasso

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2E74B5','#E74C3C','#27AE60','#F39C12','#8E44AD','#1ABC9C']

X, Y, factor_names, portfolio_names = load_all_data()
X_vals   = X.values
X_scaled = (X_vals - X_vals.mean(axis=0)) / X_vals.std(axis=0)
y_vals   = Y.iloc[:, 0].values
print('All modules loaded')

Data loaded: 288 months | 6 factors | 25 portfolios
Date range: 2000-01 to 2023-12
All modules loaded


## 6.1 Time-Series Cross-Validation

We use expanding window CV to select the optimal alpha for each method.
Key principle: training always precedes validation — no look-ahead bias.

In [3]:
alphas = [0.001, 0.003, 0.005, 0.007, 0.01, 0.02, 0.05, 0.1]

print('Cross-Validation Results:')
print()
cv_summary = {}
for name, cls, kwargs in [
    ('Ridge',      RidgeScratch,     {}),
    ('LASSO',      LassoProximal,    {}),
    ('ElasticNet', ElasticNetScratch, {'l1_ratio': 0.5})
]:
    best_alphas, median = find_best_alphas(
        X, Y, cls, alphas=alphas, **kwargs)
    cv_summary[name] = median
    print(f'{name}:')
    print(f'  Best alphas per portfolio: {best_alphas}')
    print(f'  Median best alpha: {median}')
    print()

print('Summary:')
print(f'  Ridge:      alpha = {cv_summary["Ridge"]}')
print(f'  LASSO:      alpha = {cv_summary["LASSO"]}')
print(f'  ElasticNet: alpha = {cv_summary["ElasticNet"]}')
print()
print('Ridge needs much higher alpha — consistent with L2 needing stronger')
print('regularization to prevent overfitting without sparsity')

Cross-Validation Results:

Ridge:
  Best alphas per portfolio: [0.001, 0.1, 0.001, 0.001, 0.001, 0.001, 0.1, 0.1, 0.1, 0.1, 0.001, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.001, 0.1, 0.001, 0.1, 0.1]
  Median best alpha: 0.1

LASSO:
  Best alphas per portfolio: [0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.005, 0.003, 0.003, 0.005, 0.001, 0.003, 0.003, 0.005, 0.007, 0.003, 0.005, 0.005, 0.007, 0.01, 0.001, 0.01, 0.001, 0.01, 0.005]
  Median best alpha: 0.003

ElasticNet:
  Best alphas per portfolio: [0.001, 0.003, 0.001, 0.001, 0.001, 0.001, 0.01, 0.005, 0.007, 0.01, 0.001, 0.007, 0.003, 0.007, 0.01, 0.005, 0.007, 0.01, 0.01, 0.02, 0.001, 0.02, 0.003, 0.02, 0.01]
  Median best alpha: 0.007

Summary:
  Ridge:      alpha = 0.1
  LASSO:      alpha = 0.003
  ElasticNet: alpha = 0.007

Ridge needs much higher alpha — consistent with L2 needing stronger
regularization to prevent overfitting without sparsity


## 6.2 Factor Dropout Analysis

Running LASSO across all 25 portfolios reveals which factors
are universally selected vs portfolio-specific.

In [4]:
configs = [
    ('Ridge',      RidgeScratch,     0.1,   {}),
    ('LASSO',      LassoProximal,    0.003, {}),
    ('ElasticNet', ElasticNetScratch, 0.007, {'l1_ratio': 0.5}),
]

factor_freq = {}
print('Factor dropout frequency (# portfolios where factor is DROPPED):')
print()
print(f'{"Factor":<10}', end='')
for name, _, _, _ in configs:
    print(f' {name:>12}', end='')
print()
print('-' * 46)

for name, cls, alpha, kwargs in configs:
    drop_counts = np.zeros(len(factor_names))
    for j in range(Y.shape[1]):
        m = cls(alpha=alpha, **kwargs)
        m.fit(X_scaled, Y.iloc[:, j].values)
        for i, c in enumerate(m.coef_):
            if abs(c) <= 1e-4:
                drop_counts[i] += 1
    factor_freq[name] = drop_counts

for i, fname in enumerate(factor_names):
    print(f'{fname:<10}', end='')
    for name, _, _, _ in configs:
        freq = int(factor_freq[name][i])
        print(f' {str(freq)+"/25":>12}', end='')
    print()

print()
print('KEY FINDING:')
print('  Mkt-RF: never dropped by any method — universally priced factor')
print('  CMA:    dropped by LASSO in 14/25 portfolios — weakest predictor')
print('  Mom:    dropped in 13/25 — momentum not useful for size/value sorts')
print('  Ridge:  drops almost nothing — confirms L2 does not do selection')

Factor dropout frequency (# portfolios where factor is DROPPED):

Factor            Ridge        LASSO   ElasticNet
----------------------------------------------
Mkt-RF             0/25         0/25         0/25
SMB                0/25         1/25         1/25
HML                0/25         1/25         1/25
RMW                0/25         3/25         5/25
CMA                1/25        14/25        15/25
Mom                1/25        13/25        13/25

KEY FINDING:
  Mkt-RF: never dropped by any method — universally priced factor
  CMA:    dropped by LASSO in 14/25 portfolios — weakest predictor
  Mom:    dropped in 13/25 — momentum not useful for size/value sorts
  Ridge:  drops almost nothing — confirms L2 does not do selection


## 6.3 Regime Analysis — Adaptive Lambda

We test whether scaling lambda by market volatility improves
performance during high-volatility and crisis periods.

In [5]:
# Fixed lambda
preds_f, actuals, dates, _ = walk_forward_backtest(
    X, Y, LassoProximal, alpha=0.003)
m_fixed = compute_metrics(preds_f, actuals)

# Adaptive lambda
preds_a, actuals_a, dates_a, _, lambdas_used = \
    walk_forward_backtest_adaptive(X, Y, LassoProximal, base_alpha=0.003)
m_adaptive = compute_metrics(preds_a, actuals_a)

print('Fixed vs Adaptive Lambda:')
print(f'{"Method":<22} {"OOS_R2":>8} {"Sharpe":>8} {"ICIR":>8}')
print('=' * 50)
print(f'{"Fixed Lambda":<22} {m_fixed["OOS_R2"]:>8} {m_fixed["Sharpe"]:>8} {m_fixed["ICIR"]:>8}')
print(f'{"Adaptive Lambda":<22} {m_adaptive["OOS_R2"]:>8} {m_adaptive["Sharpe"]:>8} {m_adaptive["ICIR"]:>8}')
print()
print('FINDING: Adaptive lambda does NOT improve performance')
print('INTERPRETATION: Fama-French factor structure is stable across')
print('volatility regimes — regularization does not need to adapt.')
print('This is a meaningful negative result, not a failure.')

# Regime distribution
regimes, vol = identify_regimes(X['Mkt-RF'])
print()
print('Market regime distribution (2000-2023):')
print(regimes.value_counts().to_string())

Fixed vs Adaptive Lambda:
Method                   OOS_R2   Sharpe     ICIR
Fixed Lambda             0.8983   4.9393   2.5755
Adaptive Lambda          0.8962   4.9113   2.5299

FINDING: Adaptive lambda does NOT improve performance
INTERPRETATION: Fama-French factor structure is stable across
volatility regimes — regularization does not need to adapt.
This is a meaningful negative result, not a failure.

Market regime distribution (2000-2023):
medium    103
low        92
high       87
crisis      6


## 6.4 Online Learning vs Batch

Online learning updates model incrementally — one observation at a time.
This is how production trading systems work.
We test: does online match batch performance at lower computational cost?

In [6]:
print('Running batch LASSO...')
t0 = time.time()
preds_b, actuals_b, dates_b, _ = walk_forward_backtest(
    X, Y, LassoProximal, alpha=0.003)
batch_time = time.time() - t0
m_batch    = compute_metrics(preds_b, actuals_b)

print('Running online LASSO...')
t0 = time.time()
preds_o, actuals_o, dates_o, coefs_o = walk_forward_online(
    X, Y, alpha=0.003)
online_time = time.time() - t0
m_online    = compute_metrics(preds_o, actuals_o)

print()
print('=' * 65)
print(f'{"Method":<20} {"OOS_R2":>8} {"Sharpe":>8} {"ICIR":>8} {"Time(s)":>10}')
print('=' * 65)
print(f'{"Batch LASSO":<20} {m_batch["OOS_R2"]:>8} {m_batch["Sharpe"]:>8}'
      f' {m_batch["ICIR"]:>8} {batch_time:>10.2f}')
print(f'{"Online LASSO":<20} {m_online["OOS_R2"]:>8} {m_online["Sharpe"]:>8}'
      f' {m_online["ICIR"]:>8} {online_time:>10.2f}')
print('=' * 65)
print(f'Speedup: {batch_time/online_time:.1f}x')
print()
print('KEY FINDING: Online LASSO achieves higher Sharpe and ICIR')
print('at 8x lower computational cost than batch retraining.')
print('This is directly relevant to production trading systems.')

Running batch LASSO...
Running online LASSO...

Method                 OOS_R2   Sharpe     ICIR    Time(s)
Batch LASSO            0.8983   4.9393   2.5755       1.64
Online LASSO           0.8999   5.0611   2.7023       0.19
Speedup: 8.7x

KEY FINDING: Online LASSO achieves higher Sharpe and ICIR
at 8x lower computational cost than batch retraining.
This is directly relevant to production trading systems.


## 6.5 Factor Interaction Analysis

We extend the linear model with pairwise interaction terms.
Research question: do factor interactions predict returns
beyond what linear factors capture?

In [7]:
print('Analyzing factor interactions...')
selection, coefs, feature_names, inter_names = \
    analyze_interaction_selection(
        X_scaled, Y, LassoProximal, alpha=0.003)

inter_freq = selection[:, 6:].mean(axis=0) * 100
inter_sorted = sorted(zip(inter_names, inter_freq),
                      key=lambda x: x[1], reverse=True)

print('Top interaction terms by selection frequency:')
print(f'{"Interaction":<25} {"Freq":>8} {"Economic Interpretation"}')
print('-' * 70)
interp = {
    'SMBxCMA':    'Small + conservative investment jointly predict returns',
    'RMWxMom':    'Profitable + momentum stocks compound in predictiveness',
    'SMBxMom':    'Small cap momentum effect',
    'HMLxRMW':    'Value + profitability jointly matter',
    'HMLxMom':    'Value stocks with momentum',
}
for name, freq in inter_sorted[:8]:
    note = interp.get(name, '')
    print(f'{name:<25} {freq:>7.0f}% {note}')

print()
print('NOTE: No interaction reaches 50% selection frequency')
print('Interactions are portfolio-specific, not universal')

Analyzing factor interactions...
Top interaction terms by selection frequency:
Interaction                   Freq Economic Interpretation
----------------------------------------------------------------------
SMBxCMA                        40% Small + conservative investment jointly predict returns
RMWxMom                        36% Profitable + momentum stocks compound in predictiveness
SMBxMom                        28% Small cap momentum effect
HMLxRMW                        28% Value + profitability jointly matter
HMLxMom                        28% Value stocks with momentum
CMAxMom                        24% 
Mkt-RFxSMB                     20% 
Mkt-RFxHML                     20% 

NOTE: No interaction reaches 50% selection frequency
Interactions are portfolio-specific, not universal


## 6.6 DRO vs Standard LASSO

Distributionally Robust Optimization adds a Wasserstein robustness term.
We test whether robustness improves performance, especially during crises.

In [8]:
print('Running DRO comparison...')
preds_l, actuals_l, dates_l, _ = walk_forward_backtest(
    X, Y, LassoProximal, alpha=0.003)
m_lasso = compute_metrics(preds_l, actuals_l)

print(f'{"Method":<22} {"OOS_R2":>8} {"Sharpe":>8} {"ICIR":>8}')
print('=' * 50)
print(f'{"Standard LASSO":<22} {m_lasso["OOS_R2"]:>8} {m_lasso["Sharpe"]:>8} {m_lasso["ICIR"]:>8}')

for eps in [0.01, 0.05]:
    preds_d, actuals_d, _, _ = walk_forward_backtest(
        X, Y, DROLasso, alpha=0.003, epsilon=eps)
    m_dro = compute_metrics(preds_d, actuals_d)
    name  = f'DRO (eps={eps})'
    print(f'{name:<22} {m_dro["OOS_R2"]:>8} {m_dro["Sharpe"]:>8} {m_dro["ICIR"]:>8}')
print('=' * 50)
print()
print('KEY FINDING: DRO consistently underperforms standard LASSO')
print('INTERPRETATION: FF factor relationships do not shift in ways')
print('that Wasserstein robustness addresses.')
print('The factor structure is stable — DRO conservatism is counterproductive.')

Running DRO comparison...
Method                   OOS_R2   Sharpe     ICIR
Standard LASSO           0.8983   4.9393   2.5755
DRO (eps=0.01)            0.878   4.8455   2.4725
DRO (eps=0.05)           0.6397   4.2834   1.9495

KEY FINDING: DRO consistently underperforms standard LASSO
INTERPRETATION: FF factor relationships do not shift in ways
that Wasserstein robustness addresses.
The factor structure is stable — DRO conservatism is counterproductive.


## 6.7 Signal Discovery — Raw Price Signals vs FF Factors

We test whether signals constructed from raw price/volume data
can match the predictive power of pre-built Fama-French factors.
50 stocks, 17 signals, 2000-2023.

In [9]:
# 6.7 Signal Discovery — Raw Price Signals vs FF Factors
import pickle
import warnings
warnings.filterwarnings('ignore')

print('Signal Discovery Results (from separate analysis):')
print()
print('Discovered signals selected at alpha=0.001:')
print(f'  vol_12m        +0.00451  (high long-run vol = higher return = risk premium)')
print(f'  high_52w_ratio -0.00265  (near 52w high = mean reversion)')
print(f'  vol_trend_6m   -0.00047  (rising volume = lower returns)')
print(f'  mom_6m         +0.00034  (6-month momentum positive)')
print(f'  price_ma_3m    -0.00022  (above MA = slight reversion)')
print(f'  mom_3m         +0.00021  (3-month momentum)')
print()
print('Performance comparison:')
print(f'{"Source":<25} {"Mean IC":>10} {"ICIR":>10}')
print('=' * 47)
print(f'{"Fama-French Factors":<25} {"0.6137":>10} {"2.5854":>10}')
print(f'{"Discovered Price Signals":<25} {"0.0085":>10} {"0.0364":>10}')
print('=' * 47)
print()
print('KEY FINDING: Raw price signals achieve IC=0.008 vs FF IC=0.61')
print('FF factors are 72x stronger predictors than price signals alone.')
print('Fundamental accounting data (book value, profitability, investment)')
print('contains information that price history cannot replicate.')

Signal Discovery Results (from separate analysis):

Discovered signals selected at alpha=0.001:
  vol_12m        +0.00451  (high long-run vol = higher return = risk premium)
  high_52w_ratio -0.00265  (near 52w high = mean reversion)
  vol_trend_6m   -0.00047  (rising volume = lower returns)
  mom_6m         +0.00034  (6-month momentum positive)
  price_ma_3m    -0.00022  (above MA = slight reversion)
  mom_3m         +0.00021  (3-month momentum)

Performance comparison:
Source                       Mean IC       ICIR
Fama-French Factors           0.6137     2.5854
Discovered Price Signals      0.0085     0.0364

KEY FINDING: Raw price signals achieve IC=0.008 vs FF IC=0.61
FF factors are 72x stronger predictors than price signals alone.
Fundamental accounting data (book value, profitability, investment)
contains information that price history cannot replicate.


In [10]:
from src.solvers import FISTARestart, BBLasso, CoordinateDescent
import time

methods_new = [
    ('FISTA+fn restart', FISTARestart, {'restart':'function'}),
    ('BB LASSO (alt)',   BBLasso,      {}),
    ('Coord Descent',    CoordinateDescent, {}),
]

print('New algorithms vs Proximal GD (alpha=0.003, P1):')
print(f'{"Method":<26} {"Iters":>8} {"ms":>8}')
print('-'*44)
for name, cls, kwargs in methods_new:
    t0 = time.perf_counter()
    m  = cls(alpha=0.003, **kwargs).fit(X_scaled, y_vals)
    ms = (time.perf_counter()-t0)*1000
    print(f'{name:<26} {m.n_iter_:>8} {ms:>8.2f}')

New algorithms vs Proximal GD (alpha=0.003, P1):
Method                        Iters       ms
--------------------------------------------
FISTA+fn restart                 29     1.06
BB LASSO (alt)                   13     0.30
Coord Descent                    16     0.36
